In [1]:
!pip install -q ultralytics roboflow opencv-python-headless

import torch
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))

GPU Available: True
Device Name: Tesla T4


In [2]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")
project = rf.workspace("fast-fyp").project("shelf-products-cthzl")
version = project.version(17)
dataset = version.download("yolov8")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Shelf-Products-17 in yolov8:: 100%|██████████| 15516/15516 [00:04<00:00, 3398.30it/s]


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


In [3]:
from ultralytics import YOLO

# بارگذاری مدل سبک و پرسرعت YOLOv8n (مناسب برای Edge AI و پردازش لحظه‌ای دوربین)
model = YOLO("yolov8n.pt")

# شروع آموزش مدل روی دیتاست فروشگاه
results = model.train(
    data="/content/Shelf-Products-17/data.yaml",
    epochs=25,          # ۲۵ ایپاک روی ۷۷۰۰ عکس، دقت بسیار بالایی می‌دهد و حدود ۱۵ دقیقه زمان می‌برد
    imgsz=640,
    batch=32,           # اگر خطای حافظه GPU (CUDA Out of Memory) داد، این عدد را 16 کنید
    patience=5,         # اگر تا ۵ ایپاک دقت بهتر نشد، آموزش را زودتر تمام می‌کند (Early Stopping)
    name="smart_basket_detector"
)

Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Shelf-Products-17/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=smart_basket_detector, nbs=64, nms=None, opset=None,

In [4]:
import shutil
from google.colab import files

# ساخت یک پوشه مرتب برای خروجی‌های نهایی
!mkdir -p /content/export_artifacts
!cp /content/runs/detect/smart_basket_detector/weights/best.pt /content/export_artifacts/
!cp /content/runs/detect/smart_basket_detector/results.png /content/export_artifacts/
!cp /content/runs/detect/smart_basket_detector/confusion_matrix.png /content/export_artifacts/
!cp /content/runs/detect/smart_basket_detector/val_batch0_pred.jpg /content/export_artifacts/

# زیپ کردن و دانلود روی سیستم شخصی
shutil.make_archive("/content/smart_basket_artifacts", "zip", "/content/export_artifacts")
files.download("/content/smart_basket_artifacts.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>